In [1]:
import pandas as pd

df = pd.read_csv("Processed_dataset.csv")

# ---------------------------
# 1. Identify sprained ankle groups (player-season level)
# ---------------------------
sprained_groups = df[
    df["INJURED_TYPE"].str.contains("sprained ankle", case=False, na=False)
][["PLAYER_ID", "SEASON"]].drop_duplicates()

# Merge để lấy full rows của các group này
sprained = df.merge(sprained_groups, on=["PLAYER_ID", "SEASON"], how="inner")

# ---------------------------
# 2. Duplicate và remove injury (synthetic control)
# ---------------------------
sprained_copy = sprained.copy()
sprained_copy["INJURED_TYPE"] = None
sprained_copy["INJURED_ON"] = None

# ---------------------------
# 3. Lấy nhóm hoàn toàn không bị chấn thương
# ---------------------------
no_injury_groups = df.groupby(["PLAYER_ID", "SEASON"])["INJURED_TYPE"] \
    .apply(lambda x: x.isna().all()) \
    .reset_index()

no_injury_groups = no_injury_groups[no_injury_groups["INJURED_TYPE"]]

no_injury = df.merge(
    no_injury_groups[["PLAYER_ID", "SEASON"]],
    on=["PLAYER_ID", "SEASON"],
    how="inner"
)

# ---------------------------
# 4. Combine
# ---------------------------
final_df = pd.concat(
    [sprained, sprained_copy, no_injury],
    ignore_index=True
)

# Optional: shuffle
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Save
final_df.to_csv("ankle_injuries.csv", index=False)

In [6]:
# Keep only Knee_injury rows and rows with missing injury type
df = df[(df["INJURED_TYPE"] == "Knee_injury") | (df["INJURED_TYPE"].isna())]

# 1. Identify Knee_injury groups (exact match)
knee_groups = df[
    df["INJURED_TYPE"] == "Knee_injury"
][["PLAYER_ID", "SEASON"]].drop_duplicates()

knee = df.merge(knee_groups, on=["PLAYER_ID", "SEASON"], how="inner")

# 2. Duplicate và remove injury
knee_copy = knee.copy()
knee_copy["INJURED_TYPE"] = None
knee_copy["INJURED_ON"] = None

# 3. Nhóm hoàn toàn không bị chấn thương
no_injury_groups = df.groupby(["PLAYER_ID", "SEASON"])["INJURED_TYPE"] \
    .apply(lambda x: x.isna().all()) \
    .reset_index()

no_injury_groups = no_injury_groups[no_injury_groups["INJURED_TYPE"]]

no_injury = df.merge(
    no_injury_groups[["PLAYER_ID", "SEASON"]],
    on=["PLAYER_ID", "SEASON"],
    how="inner"
)

# 4. Combine
final_knee_df = pd.concat(
    [knee, knee_copy, no_injury],
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

final_knee_df.to_csv("knee_injuries.csv", index=False)